# Piedra, Papel o Tijera: entrenamiento y exportación a LiteRT

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tencioeduardo/mini-taller-liteRT/blob/main/Demostraci%C3%B3n/entrenar_rps.ipynb)

Este cuaderno cubre la parte del flujo de LiteRT que ocurre **fuera** del dispositivo (diapositivas 3, 7 y 8):

| Paso | Qué se hace | Diapositiva |
|---|---|---|
| Entrenar | CNN pequeña con Keras sobre el dataset `rock_paper_scissors` | 2 |
| Convertir | Exportar el modelo al formato `.tflite` (FlatBuffer) | 3, 5 |
| Comprimir | Generar **cuatro variantes**: FP32, FP16, INT8 dinámico e INT8 completo (PTQ con calibración) | 7, 8 |
| Verificar | Medir la precisión de cada variante con el runtime **LiteRT** (`ai-edge-litert`), no con `tf.lite` | 3 |
| Exportar | Descargar los modelos y el set de prueba para medirlos en la laptop con `evaluar.py` | 11, 13 |

**Salida:** `artefactos_rps.zip`, que se descomprime dentro de la carpeta `demostracion/` del repositorio:

```text
modelos/  rps_fp32.tflite  rps_fp16.tflite  rps_int8_dyn.tflite  rps_int8_full.tflite  labels.txt  manifest.json
datos/test/  Piedra/  Papel/  Tijera/   (imágenes JPEG del split de prueba)
```

**Cómo ejecutarlo:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU* (opcional, acelera el entrenamiento) y luego *Entorno de ejecución → Ejecutar todas*.

## 1. Dependencias

`tensorflow_datasets` importa `importlib_resources` (a través de su dependencia `etils`) pero no lo declara, y el entorno actual de Colab no lo trae instalado. Sin esta instalación, `tfds.load` falla con `ModuleNotFoundError: No module named 'importlib_resources'`.

`%pip` es un comando de cuaderno: funciona en una celda, pero **no** dentro de un archivo `.py` ejecutado con `!python`. Por eso este flujo vive en un cuaderno.

También se instala `ai-edge-litert`, el runtime oficial de LiteRT, para verificar los modelos con el mismo motor que se usará en la laptop. La versión se fija para que coincida con `requirements.txt`.

In [ ]:
%pip install -q importlib-resources ai-edge-litert==2.3.0

## 2. Configuración y versiones

Las versiones se registran en `manifest.json` para poder rastrear con qué herramientas se generó cada modelo.

In [ ]:
import datetime
import importlib.metadata as md
import json
import os
import shutil

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

IMG = 150                 # tamaño de entrada del modelo: 150x150x3
BATCH = 64
EPOCHS = 12
N_REPRESENTATIVO = 200    # muestras para calibrar la cuantización INT8 completa (diapositiva 8: ~100 a 500)
SEMILLA = 42

SALIDA = "artefactos"
DIR_MODELOS = os.path.join(SALIDA, "modelos")
DIR_TEST = os.path.join(SALIDA, "datos", "test")

# Reproducibilidad: misma semilla para TensorFlow, NumPy y Python.
tf.keras.utils.set_random_seed(SEMILLA)

VERSIONES = {
    "tensorflow": tf.__version__,
    "tensorflow_datasets": tfds.__version__,
    "ai_edge_litert": md.version("ai-edge-litert"),
    "numpy": np.__version__,
}
print(json.dumps(VERSIONES, indent=2))
print("GPU disponible:", bool(tf.config.list_physical_devices("GPU")))

## 3. Dataset

`rock_paper_scissors` contiene imágenes de 300×300 de manos **generadas por computadora** sobre fondo blanco (2520 de entrenamiento y 372 de prueba). Licencia CC BY 2.0, autor Laurence Moroney.

El orden de las clases en el dataset es `rock, paper, scissors`, y se traduce a `Piedra, Papel, Tijera`. La verificación de abajo detiene el cuaderno si ese orden cambiara.

El aumento de datos (recortes, brillo, contraste y saturación aleatorios) ayuda a que el modelo tolere mejor una cámara real, cuyo fondo e iluminación difieren del dataset sintético.

In [ ]:
(ds_train, ds_test), info = tfds.load(
    "rock_paper_scissors", split=["train", "test"], as_supervised=True, with_info=True)

CLASES_DATASET = info.features["label"].names
assert CLASES_DATASET == ["rock", "paper", "scissors"], f"Orden de clases inesperado: {CLASES_DATASET}"
ETIQUETAS = ["Piedra", "Papel", "Tijera"]
print("Clases:", dict(zip(CLASES_DATASET, ETIQUETAS)))
print("Entrenamiento:", info.splits["train"].num_examples, "| Prueba:", info.splits["test"].num_examples)


def prep_eval(img, lbl):
    # Mismo rango que usará la demo: float32 en [0, 1] (--norm 0-1).
    img = tf.image.resize(img, (IMG, IMG))
    return tf.cast(img, tf.float32) / 255.0, lbl


def prep_train(img, lbl):
    img = tf.image.random_flip_left_right(img)
    lado = tf.random.uniform([], 220, 301, dtype=tf.int32)                 # zoom aleatorio
    img = tf.image.random_crop(img, tf.stack([lado, lado, 3]))
    img = tf.cast(tf.image.resize(img, (IMG, IMG)), tf.float32) / 255.0
    img = tf.image.random_brightness(img, 0.25)
    img = tf.image.random_contrast(img, 0.7, 1.3)
    img = tf.image.random_saturation(img, 0.7, 1.3)
    return tf.clip_by_value(img, 0.0, 1.0), lbl


AUTOTUNE = tf.data.AUTOTUNE
train = ds_train.shuffle(1000, seed=SEMILLA).map(prep_train, AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)
test = ds_test.map(prep_eval, AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)

## 4. Modelo y entrenamiento

Una CNN pequeña: cuatro bloques convolución + *max pooling*, una capa densa y una salida *softmax* de 3 clases. Es deliberadamente simple: el objetivo del taller es el despliegue, no la arquitectura.

In [ ]:
L = tf.keras.layers
model = tf.keras.Sequential([
    L.Input((IMG, IMG, 3)),
    L.Conv2D(32, 3, activation="relu"), L.MaxPooling2D(),
    L.Conv2D(64, 3, activation="relu"), L.MaxPooling2D(),
    L.Conv2D(128, 3, activation="relu"), L.MaxPooling2D(),
    L.Conv2D(128, 3, activation="relu"), L.MaxPooling2D(),
    L.Flatten(), L.Dropout(0.5),
    L.Dense(128, activation="relu"),
    L.Dense(3, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()
historial = model.fit(train, epochs=EPOCHS, validation_data=test)

In [ ]:
# Precisión del modelo Keras original: es la referencia antes de convertir.
_, PRECISION_KERAS = model.evaluate(test, verbose=0)
print(f"Precisión Keras (referencia) en el set de prueba: {PRECISION_KERAS:.2%}")

# SavedModel: formato intermedio desde el que convierte TFLiteConverter.
model.export("rps_saved_model")

## 5. Conversión y cuantización: cuatro variantes

Las cuatro variantes salen del **mismo** modelo entrenado, así que cualquier diferencia de tamaño, latencia o precisión se debe solo a la conversión.

| Variante | Pesos | Activaciones | Calibración | Diapositiva |
|---|---|---|---|---|
| `rps_fp32` | float32 | float32 | no | 7 (referencia) |
| `rps_fp16` | float16 | float32 | no | 7, 11 |
| `rps_int8_dyn` | int8 | float32 (se cuantizan "al vuelo") | no | 8 (PTQ sin dataset) |
| `rps_int8_full` | int8 | int8 | sí, con 200 muestras | 8 (PTQ con dataset representativo) |

En `rps_int8_full`, la entrada y la salida se dejan en float32 (el conversor inserta nodos de cuantizar y des-cuantizar en los extremos). Así el modelo funciona con las dos APIs de LiteRT: el `TensorBuffer` de `CompiledModel` en Python no admite tensores uint8.

In [ ]:
os.makedirs(DIR_MODELOS, exist_ok=True)


def dataset_representativo():
    # Imágenes reales del set de ENTRENAMIENTO (sin aumento de datos), una por lote.
    # El conversor las usa para observar el rango real de cada activación y elegir
    # su escala y punto cero: valor_real = escala * (entero - punto_cero).
    for img, _ in ds_train.map(prep_eval).take(N_REPRESENTATIVO):
        yield [img[None, ...]]


def convertir(variante):
    conv = tf.lite.TFLiteConverter.from_saved_model("rps_saved_model")
    if variante == "fp16":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.target_spec.supported_types = [tf.float16]
    elif variante == "int8_dyn":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
    elif variante == "int8_full":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.representative_dataset = dataset_representativo
        conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    contenido = conv.convert()
    # Un .tflite es un FlatBuffer con identificador "TFL3" en los bytes 4-7.
    assert contenido[4:8] == b"TFL3", f"{variante}: el resultado no es un FlatBuffer TFLite"
    return contenido


VARIANTES = ["fp32", "fp16", "int8_dyn", "int8_full"]
ARCHIVOS = {}
for v in VARIANTES:
    ruta = os.path.join(DIR_MODELOS, f"rps_{v}.tflite")
    with open(ruta, "wb") as f:
        f.write(convertir(v))
    ARCHIVOS[v] = ruta

tam_fp32 = os.path.getsize(ARCHIVOS["fp32"])
print(f"{'Variante':<12}{'Tamaño':>12}{'vs FP32':>10}")
for v, ruta in ARCHIVOS.items():
    t = os.path.getsize(ruta)
    print(f"{v:<12}{t / 1024:>9.0f} KiB{t / tam_fp32:>9.0%}")

## 6. Verificación con el runtime LiteRT

Se mide la precisión de cada `.tflite` con `ai_edge_litert.interpreter.Interpreter`, el mismo runtime que usa la laptop. No se usa `tf.lite.Interpreter`: está obsoleto desde TensorFlow 2.19 y su eliminación está anunciada.

Además de la precisión, se reporta qué porcentaje de predicciones **coincide con FP32**. Esa métrica aísla el efecto de la cuantización: si una variante coincide al 100 % con FP32, la compresión no cambió ninguna decisión del modelo.

> La precisión que mida `evaluar.py` en la laptop puede diferir unas décimas: allí las imágenes se leen desde JPEG y se redimensionan con OpenCV (el mismo camino que la cámara), mientras que aquí se usa `tf.image.resize`. Medir con el pipeline real del dispositivo es justamente el punto de la diapositiva 11.

In [ ]:
from ai_edge_litert.interpreter import Interpreter

x_test, y_test = [], []
for x, y in test.unbatch():
    x_test.append(x.numpy()); y_test.append(int(y))
x_test, y_test = np.stack(x_test), np.array(y_test)


def predecir_todo(ruta):
    it = Interpreter(model_path=ruta)
    it.allocate_tensors()
    i, o = it.get_input_details()[0], it.get_output_details()[0]
    assert i["dtype"] == np.float32, "Se esperaba entrada float32 en todas las variantes"
    pred = []
    for x in x_test:
        it.set_tensor(i["index"], x[None].astype(np.float32))
        it.invoke()
        pred.append(int(np.argmax(it.get_tensor(o["index"]))))
    return np.array(pred)


PRED = {v: predecir_todo(r) for v, r in ARCHIVOS.items()}
RESUMEN = {}
print(f"{'Variante':<12}{'Precisión':>11}{'Coincide FP32':>15}")
for v in VARIANTES:
    prec = float((PRED[v] == y_test).mean())
    coinc = float((PRED[v] == PRED["fp32"]).mean())
    RESUMEN[v] = {"precision": prec, "coincidencia_fp32": coinc}
    print(f"{v:<12}{prec:>11.2%}{coinc:>15.2%}")

## 7. Etiquetas, manifiesto y set de prueba

- `labels.txt`: una etiqueta por línea, en el orden de las salidas del modelo.
- `manifest.json`: versiones, configuración, tamaños y precisión de cada variante. Es la trazabilidad de los modelos.
- `datos/README.md`: atribución y licencia del dataset (CC BY 2.0).
- `datos/test/<Etiqueta>/*.jpg`: el split de prueba completo, en su tamaño original (300×300), para que `evaluar.py` mida la precisión en la laptop **sin instalar TensorFlow**. Calidad JPEG 95.

In [ ]:
with open(os.path.join(DIR_MODELOS, "labels.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(ETIQUETAS) + "\n")

if os.path.isdir(DIR_TEST):
    shutil.rmtree(DIR_TEST)
for e in ETIQUETAS:
    os.makedirs(os.path.join(DIR_TEST, e), exist_ok=True)
conteo = dict.fromkeys(ETIQUETAS, 0)
for n, (img, lbl) in enumerate(ds_test):
    etiqueta = ETIQUETAS[int(lbl)]
    jpeg = tf.io.encode_jpeg(img, quality=95).numpy()
    with open(os.path.join(DIR_TEST, etiqueta, f"{n:04d}.jpg"), "wb") as f:
        f.write(jpeg)
    conteo[etiqueta] += 1
print("Imágenes de prueba exportadas:", conteo)

# Atribución del dataset (licencia CC BY 2.0), versionada junto a las imágenes.
with open(os.path.join(SALIDA, "datos", "README.md"), "w", encoding="utf-8") as f:
    f.write(
        "# Set de prueba: rock_paper_scissors\n\n"
        f"Split `test` completo ({sum(conteo.values())} imágenes de 300x300, JPEG calidad 95), "
        "exportado por `entrenar_rps.ipynb`.\n"
        "Carpetas: Piedra = rock, Papel = paper, Tijera = scissors.\n\n"
        "**Fuente:** dataset *Rock Paper Scissors* de Laurence Moroney, distribuido en "
        "TensorFlow Datasets (https://www.tensorflow.org/datasets/catalog/rock_paper_scissors).\n"
        "**Licencia:** Creative Commons Attribution 2.0 (CC BY 2.0). "
        "Las imágenes se redistribuyen sin modificaciones de contenido (solo recodificadas a JPEG).\n")

manifiesto = {
    "generado": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "versiones": VERSIONES,
    "dataset": {"nombre": "rock_paper_scissors", "licencia": "CC BY 2.0 (Laurence Moroney)",
                "imagenes_prueba": conteo},
    "entrenamiento": {"epocas": EPOCHS, "lote": BATCH, "semilla": SEMILLA,
                      "precision_keras": float(PRECISION_KERAS)},
    "entrada": {"forma": [1, IMG, IMG, 3], "tipo": "float32", "normalizacion": "0-1"},
    "etiquetas": ETIQUETAS,
    "modelos": {
        f"rps_{v}": {
            "archivo": os.path.basename(ARCHIVOS[v]),
            "tamano_bytes": os.path.getsize(ARCHIVOS[v]),
            "precision_colab": RESUMEN[v]["precision"],
            "coincidencia_fp32_colab": RESUMEN[v]["coincidencia_fp32"],
        } for v in VARIANTES
    },
}
with open(os.path.join(DIR_MODELOS, "manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifiesto, f, indent=2, ensure_ascii=False)
print(json.dumps(manifiesto["modelos"], indent=2))

## 8. Descarga

Descomprima `artefactos_rps.zip` **dentro** de la carpeta `demostracion/` del repositorio; se crearán `modelos/` y `datos/`. Luego, en la laptop:

```bash
./setup.sh                 # valida los modelos y el set de prueba
python evaluar.py          # Acto 2 de la demo: tamaño, latencia, precisión y RAM
python litert_rps.py       # Acto 3: cámara en vivo
```

In [ ]:
archivo_zip = shutil.make_archive("artefactos_rps", "zip", SALIDA)
print(f"Generado: {archivo_zip} ({os.path.getsize(archivo_zip) / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(archivo_zip)
except ImportError:
    print("Fuera de Colab: copie el zip manualmente a la carpeta demostracion/.")